In [ ]:
base = "/Volumes/cs483eviction/raw/lcbh_raw"
files = [f.name for f in dbutils.fs.ls(base)]

raw = {}
for name in files:
    raw[name] = (spark.read.option("header", True)
                      .option("inferSchema", True)
                      .csv(f"{base}/{name}"))
    print(f"=== {name}: {raw[name].count()} rows ===")


In [ ]:
from pyspark.sql import functions as F

ACS_SPLIT_YEAR = 2012
COVID_START, COVID_END = 2020, 2022

ev = (raw["eviction_data_tract.csv"]
      .withColumn("tract_id", F.lpad(F.col("tract").cast("string"), 11, "0"))
      .withColumnRenamed("filing_year", "year")
      .withColumn("count_missing", F.col("eviction_filings_total").isNull())
      .withColumn("period",
          F.when(F.col("year") < COVID_START, "pre_covid")
           .when(F.col("year").between(COVID_START, COVID_END), "covid")
           .otherwise("post_covid"))
      .dropDuplicates(["tract_id", "year"]))

cen = (raw["census_data_tract.csv"]
       .withColumn("tract_id", F.lpad(F.col("tract").cast("string"), 11, "0"))
       .withColumn("acs_key", F.regexp_extract("census_year", r"^(\d{4}-\d{4})", 1))
       .drop("tract", "census_year"))

ev2 = ev.withColumn("acs_key",
        F.when(F.col("year") <= ACS_SPLIT_YEAR, "2006-2010").otherwise("2014-2018"))


In [ ]:
print("duplicate census keys:",
      cen.groupBy("tract_id", "acs_key").count().filter("count > 1").count())

final = (ev2.join(cen, ["tract_id", "acs_key"], "left")
   .withColumn("eviction_rate_per_100",
       F.when(F.col("housing_units_rental") > 0,
              F.col("eviction_filings_total") / F.col("housing_units_rental") * 100))
   .withColumn("eviction_rate_per_1000",
       F.when(F.col("housing_units_rental") > 0,
              F.col("eviction_filings_total") / F.col("housing_units_rental") * 1000))
   .drop("tract"))

print("eviction rows:", ev.count(), "| final rows:", final.count())


In [ ]:
display(final.filter("eviction_filings_rate is not null")
             .select("tract_id", "year", "eviction_filings_rate",
                     "eviction_rate_per_100", "eviction_rate_per_1000")
             .limit(20))


In [ ]:
(final.write.format("delta")
      .mode("overwrite")
      .saveAsTable("cs483eviction.default.lcbh_tract_year"))

print("saved rows:", spark.table("cs483eviction.default.lcbh_tract_year").count())


In [ ]:
comm = raw["eviction_data_comm_area.csv"]

display(comm.filter(F.col("filing_year") == 2019)
            .orderBy(F.desc("eviction_filings_total"))
            .select("area_name", "eviction_filings_total", "eviction_filings_rate")
            .limit(10))

display(comm.filter(F.col("filing_year") == 2019)
            .orderBy(F.desc("eviction_filings_rate"))
            .select("area_name", "eviction_filings_total", "eviction_filings_rate")
            .limit(5))


In [ ]:
df = spark.table("cs483eviction.default.lcbh_tract_year").toPandas()
df.to_csv("/Volumes/cs483eviction/default/processed/lcbh_tract_year.csv", index=False)
print("exported", len(df), "rows")
